# Day 052 — Exercise 2: Request & Response Models

**What you'll build:** the Pydantic models `ChatRequest` and `ChatResponse`, plus `create_echo_app()` — a `POST /echo` endpoint that validates the request body and echoes it back as a typed response.

**Why it matters:** An API is a contract. FastAPI uses Pydantic models (Day 4) as that contract: declare `ChatRequest` as the parameter type and FastAPI parses the JSON body, validates it, and returns a **422** automatically on bad input — before your code runs. `response_model` does the same on the way out. You write the shapes; FastAPI enforces them.

## Provided: Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient
import ollama

## Your Implementation

In [ ]:
class ChatRequest(BaseModel):
    """Request body for the chat endpoints."""
    # TODO: message: str = Field(min_length=1, description='User message for the model')
    # TODO: temperature: float = Field(default=0.7, ge=0.0, le=1.0)
    pass


class ChatResponse(BaseModel):
    """Response body returned by the chat endpoints."""
    # TODO: reply: str
    # TODO: model: str
    pass


def create_echo_app() -> FastAPI:
    """POST /echo — validate a ChatRequest, echo it as a ChatResponse (no model call)."""
    app = FastAPI()

    # TODO: @app.post('/echo', response_model=ChatResponse)
    #       def echo(req: ChatRequest):
    #           return ChatResponse(reply=req.message, model='echo')

    return app

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: models validate a good dict (Pydantic v2)
    try:
        req = ChatRequest.model_validate({'message': 'hi', 'temperature': 0.5})
        assert req.message == 'hi' and req.temperature == 0.5
        assert ChatRequest.model_validate({'message': 'x'}).temperature == 0.7, 'temperature default should be 0.7'
        passed += 1; print('✅ Check 1: ChatRequest validates + defaults temperature')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: POST /echo with a valid body returns 200 and echoes the message
    try:
        client = TestClient(create_echo_app())
        r = client.post('/echo', json={'message': 'hello api'})
        assert r.status_code == 200, f'expected 200, got {r.status_code}'
        assert r.json()['reply'] == 'hello api', f"bad echo: {r.json()}"
        passed += 1; print('✅ Check 2: POST /echo echoes the message (200)')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: missing required field -> automatic 422
    try:
        client = TestClient(create_echo_app())
        r = client.post('/echo', json={'temperature': 0.5})
        assert r.status_code == 422, f'expected 422 on missing message, got {r.status_code}'
        passed += 1; print('✅ Check 3: missing message -> 422')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: empty message violates min_length -> 422
    try:
        client = TestClient(create_echo_app())
        r = client.post('/echo', json={'message': ''})
        assert r.status_code == 422, f'expected 422 on empty message, got {r.status_code}'
        passed += 1; print('✅ Check 4: empty message -> 422 (min_length)')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: temperature out of range -> 422
    try:
        client = TestClient(create_echo_app())
        r = client.post('/echo', json={'message': 'hi', 'temperature': 2.0})
        assert r.status_code == 422, f'expected 422 on temperature=2.0, got {r.status_code}'
        passed += 1; print('✅ Check 5: temperature out of [0,1] -> 422')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
class ChatRequest(BaseModel):
    """Request body for the chat endpoints."""
    message: str = Field(min_length=1, description='User message for the model')
    temperature: float = Field(default=0.7, ge=0.0, le=1.0)


class ChatResponse(BaseModel):
    """Response body returned by the chat endpoints."""
    reply: str
    model: str


class HealthResponse(BaseModel):
    """Response body for the health check."""
    status: str
    model: str


def create_echo_app() -> FastAPI:
    """POST /echo — validate a ChatRequest and echo it back as a ChatResponse,
    WITHOUT calling the model. Deterministic, so it tests the request/response
    contract (and FastAPI's automatic 422 on bad input) with no LLM involved.
    """
    app = FastAPI()

    @app.post('/echo', response_model=ChatResponse)
    def echo(req: ChatRequest):
        return ChatResponse(reply=req.message, model='echo')

    return app
```

**Why this works:** Declaring `def echo(req: ChatRequest)` tells FastAPI the JSON body must match `ChatRequest`. `Field(min_length=1)` and `Field(ge=0.0, le=1.0)` are Pydantic v2 constraints — violate any of them and FastAPI returns a 422 with a precise error list, before `echo` ever runs. `response_model=ChatResponse` validates and shapes the output too. The endpoint is deterministic (no model call), so it's the perfect place to prove the request/response contract.
</details>